<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/notes/6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №6: Синтез цифровых фильтров

## Типы цифровых фильтров

**КИХ (FIR)** — конечная импульсная характеристика (Finite Impulse Response)
- Всегда устойчив, проще реализация
- Можно добиться строго линейной фазы
- Требует большего порядка для резких срезов
- Разностное уравнение:
$$
y[n] = \sum_{k=0}^{M} b_k x[n-k]
$$

**БИХ (IIR)** — бесконечная импульсная характеристика (Infinite Impulse Response)
- Может быть неустойчивым
- Нелинейная фаза (искажает форму сигнала)
- Более эффективен (меньше вычислений)
- Разностное уравнение:
$$
y[n] = \sum_{k=0}^{M} b_k x[n-k] - \sum_{k=1}^{N} a_k y[n-k]
$$

---

## Основные характеристики фильтров

**АЧХ** — амплитудно-частотная характеристика (показывает ослабление сигнала на разных частотах)

**ФЧХ** — фазо-частотная характеристика (показывает сдвиг фазы)

**Групповая задержка** — производная ФЧХ по частоте. Константа ⇒ линейная фаза, нет искажений формы сигнала

**Параметры:**
- **fc** — частота среза (граница между полосой пропускания и полосой заграждения)
- **N** — порядок фильтра или длина КИХ
- **Пульсации** — неравномерность АЧХ в полосах
- **Переходная полоса** — область между ПП и ПЗ

---

## Оконный метод синтеза КИХ-фильтров

`scipy.signal.firwin(numtaps, cutoff, window='hamming', fs=None)`

**Параметры:**
- `numtaps` — длина фильтра (нечётное)
- `cutoff` — частота среза (по уровню $-6$ дБ), в тех же единицах, что и `fs`
- `window` — тип окна: 'boxcar', 'hann', 'hamming', 'blackman' и т. д.
- `fs` — частота дискретизации
- Возвращает массив коэффициентов `b`


## Равноволновой метод

`scipy.signal.remez(numtaps, bands, desired, weight=None, fs=None)`

**Параметры:**
- `numtaps` — длина фильтра
- `bands` — массив границ ПП/ПЗ в тех же единицах, что и `fs` (например, `[0, 150, 250, 500]`)
- `desired` — желаемое усиление в полосах (`[1, 0]`)
- `weight` — веса полос (влияют на пульсации)
- `fs` — частота дискретизации
- Возвращает массив коэффициентов `b`

**Смысл весов:**
Увеличение веса для полосы заграждения ⇒ глубже подавление, но больше пульсации в ПП (и наоборот).

---

## БИХ-фильтры: Баттерворт

`scipy.signal.butter(N, Wn, btype='low', output='ba', fs=None)`

**Параметры:**
- `N` — порядок фильтра
- `Wn` — частота среза (по уровню $-3 дБ$), в тех же единицах, что и `fs`, или нормированная от 0 до 1, если `fs=None`
- `btype` — тип фильтра ('low', 'high', 'band', 'stop')
- `output` — формат вывода ('ba', 'zpk', 'sos'), рекомендуется 'sos'

---

## БИХ-фильтры: Чебышев тип I

`scipy.signal.cheby1(N, rp, Wn, btype='low', output='ba', fs=None)`

**Параметры:**
- `rp` — максимально разрешённая амплитуда пульсаций в ПП (дБ, положительное число)
- Остальные — как у `butter`

**Особенности:**
- Равновеликие пульсации в ПП, монотонный спад в ПЗ
- Более крутой спад, чем у Баттерворта того же порядка
- Цена — пульсации в ПП (задаются `rp`)
- Частота Wn — граница ПП, где пропускание падает ниже `-rp`

---

## БИХ-фильтры: Чебышев тип II

`scipy.signal.cheby2(N, rs, Wn, btype='low', output='ba', fs=None)`

**Параметры:**
- `rs` — минимальное затухание, требуемое в ПЗ (дБ, положительное число)
- Остальные — как у `butter`

**Особенности:**
- Монотонная АЧХ в ПП, равновеликие пульсации в ПЗ
- Подавление в ПЗ не хуже rs
- Частота Wn — граница ПЗ, где затухание достигает rs

---

## Эллиптический фильтр (Кауэра)

`scipy.signal.ellip(N, rp, rs, Wn, btype='low', output='ba', fs=None)`

**Параметры:**
- `rp` — пульсации в ПП (дБ)
- `rs` — затухание в ПЗ (дБ)
- Остальные — как у `butter`

**Особенности:**
- Пульсации и в ПП, и в ПЗ
- Самый крутой спад при заданном порядке
- Максимальная эффективность, но сильные фазовые искажения
- Частота Wn — граница ПП, где пропускание падает ниже `-rp`

**Примечание:** при rp→0 становится Чебышевым II, при rs→0 — Чебышевым I, при rp, rs→0 — Баттервортом

---

## Анализ частотных характеристик

`scipy.signal.freqz(b, a=1, worN=512, fs=None)`
`scipy.signal.freqz_sos(sos, worN=512, fs=None)`

**Параметры:**
- `b`, `a` — коэффициенты фильтра
- `sos` — коэффициенты фильтра в формате 'sos' (second-order sections)
- `worN` — число точек или массив частот
- `fs` — частота дискретизации

**Возвращает:**
- `w` — массив частот
- `h` — комплексная частотная характеристика

АЧХ в дБ: `mag_dB = 20 * np.log10(np.abs(h) + 1e-12)`

---

## Групповая задержка

`scipy.signal.group_delay(system, w=512, fs=None)`

**Что это?**
Групповая задержка показывает, на сколько отсчётов задерживаются огибающие спектральных компонент сигнала

**Параметры:**
- `system` — кортеж `(b, a)` (из формата 'sos' можно получить функцией `scipy.signal.sos2tf`)
- `w` — число точек/массив частот
- `fs` — частота дискретизации

**Важно:** Константная групповая задержка ⇒ линейная фаза, форма сигнала не искажается.

---

## Фильтрация сигналов

### Прямая фильтрация
`scipy.signal.lfilter(b, a, x)`
`scipy.signal.sosfilt(sos, x)`
- Обычный цифровой фильтр (есть задержка, нелинейная фаза для БИХ)
- Применим для реального времени

### Двухсторонняя фильтрация
`scipy.signal.filtfilt(b, a, x)`
`scipy.signal.sosfiltfilt(sos, x)`
- Двунаправленная обработка (вперёд+назад)
- Нулевая фаза, сигнал не смещается по времени
- Только для офлайн-обработки
